# GRACE reproduced code-based — contract v2

Đối chứng theo contract cho devign. Đọc README và `RESEARCH.md` trước khi so sánh. Đây là tái lập có các thích nghi đã công khai; không phải khôi phục bit-for-bit run tác giả.

Notebook dùng Python/WSL hiện có, Joern và CodeT5 local. Các stage ghi checkpoint/manifest riêng; hai profile dùng chung cache đã xác minh. Cấu hình hiện tại chạy full test, prepare, freeze và inference Vertex AI với tối đa 10 request đồng thời; Joern dùng 4 worker.

In [1]:
from pathlib import Path
import sys, json, os
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / "Graduation-Project/Ablation_V2/Reproduced-Code-Based/reproduction_pipeline.py").is_file()), None)
if ROOT is None:
    raise FileNotFoundError("Could not locate workspace root: expected Graduation-Project/Ablation_V2/Reproduced-Code-Based/reproduction_pipeline.py above the current directory")
sys.path.insert(0, str(ROOT / "Graduation-Project/Ablation_V2/Reproduced-Code-Based"))
import importlib
import reproduction_pipeline
reproduction_pipeline = importlib.reload(reproduction_pipeline)
Config, Pipeline = reproduction_pipeline.Config, reproduction_pipeline.Pipeline

DATASET = 'devign'
SPLIT = "test"  # full frozen test cohort
MODEL_DIR = str(ROOT / (
    "Graduation-Project/Ablation_V2/Reproduced-Code-Based/"
    "kaggle_gpu_job/downloads/"
    "quangdung1111-grace-kaggle-gpu-ipynb/downloads/model"
))
BATCH_SIZE = 4  # lower peak RAM for CodeT5
WORKERS = 4  # up to four Joern processes at a time
INFERENCE_WORKERS = 10  # concurrent Vertex AI requests; independent of Joern
JOERN_SHARD_INDEX = int(os.environ.get("JOERN_SHARD_INDEX", "0"))
JOERN_SHARD_COUNT = int(os.environ.get("JOERN_SHARD_COUNT", "1"))
MAX_TARGETS = None  # full train and full frozen test cohort
RUN_PREPARE = True
FREEZE = True
RUN_INFERENCE = True
if JOERN_SHARD_COUNT > 1:
    FREEZE = False
    RUN_INFERENCE = False  # shards only build Joern graph caches
config = Config(variant='code', dataset=DATASET, split=SPLIT,
                root=ROOT, model_dir=MODEL_DIR, max_targets=MAX_TARGETS, batch_size=BATCH_SIZE,
                workers=WORKERS, inference_workers=INFERENCE_WORKERS)
pipeline = Pipeline(config)
print("Python:", sys.executable)
print("Joern:", os.environ.get("JOERN_HOME"))
print("Output:", pipeline.output)


Python: /usr/bin/python3
Joern: /home/yungn/tools/joern/y/joern-cli
Output: /mnt/c/MY-FILES/LAB-THAY-THO/Graduation-Project/Ablation_V2/Reproduced-Code-Based/devign/test


## Cách chạy

Chọn **kernel venv WSL hiện có → Restart Kernel → Run All**. Notebook chạy trực tiếp trong venv được chọn; cell cấu hình in đường dẫn Python và Joern để đối chiếu. Cấu hình `.env` được nạp tự động. Không cần khởi động batch để chạy notebook thủ công. Để chia Joern qua nhiều máy, đặt cùng JOERN_SHARD_COUNT=N và mỗi máy dùng JOERN_SHARD_INDEX khác nhau từ 0 đến N-1. Mỗi máy cần checkout/cache riêng. Run All khi shard count > 1 chỉ tạo cache shard, tự tắt freeze và inference; gom các joern_shard_*.zip về một máy, merge bằng stage merge-joern-shards rồi chạy prepare đầy đủ không shard.

Cache/checkpoint hợp lệ được giữ và tái sử dụng khi Run All lại; `RUN_INFERENCE=True` gọi Vertex thật sau prepare/freeze với tối đa 10 request đồng thời. Joern giữ tối đa 4 worker.

## 1. Audit data và khả năng tái sử dụng

Xác minh SHA-256 parquet, cohort test đã freeze, ID/code hash, full train và source code cạnh CPG binary. Pool loại mọi bản sao code mục tiêu. ReVeal validation được tạo bằng adapter canonical, Devign validation giữ cohort của Full.

In [2]:
audit = pipeline.audit()
print(json.dumps(audit, ensure_ascii=False, indent=2))


{
  "schema": "grace_reproduction_v1",
  "variant": "code",
  "dataset": "devign",
  "split": "test",
  "contract_sha256": "becd38a9afd9677f2cdee47b1669f0442e53dd82c4b001c137d90d83a6710b3c",
  "dataset_manifest_sha256": "c1f40056953fa3062867e0e9dfdf1dda2a1f7fcfd9c37ffab9a0f0cba1fc31cc",
  "target_count": 2732,
  "train_count": 21854,
  "target_identity": "723786f19feb026da6b77aa18a152c070eac04b3daa1a60a6bad2d0bd5c5edd4",
  "train_identity": "8ad43a0588fc996e88d3a5a806c032db5b0f4d16eaff3e547f3f9e30089f3baa",
  "train_target_exact_code_overlap": 4,
  "same_code_candidates_excluded": true,
  "graph_reuse": {
    "graph_index": "/mnt/c/MY-FILES/LAB-THAY-THO/Graduation-Project/Ablation_V2/Devign/Code-CPG/devign_test_reextracted/graphs.jsonl",
    "graph_index_sha256": "5ef7880e2150fa4bfca16359d20d2050db35961e1823444f79c5ecaef8850718",
    "target_count": 2732,
    "approved_target_records": 2732,
    "approved_unique_code_binaries": 2731,
    "rejection_reasons": {},
    "recovered_missing_

## 2. Chuẩn bị retrieval, whole CPG và prompt

Đặt `RUN_PREPARE=True` ở cell cấu hình rồi chạy lại. Pipeline dùng cùng initial top-50 với Full, chọn semantic top-5, rerank lexical/syntactic rồi lấy một demo train có nhãn. Profile code: whitening/IP và sequence ratio trên AST edge strings; profile paper: t-SNE/L2 và Joern AST SimSBT. CodeT5 checkpoint và các lựa chọn paper chưa quy định được ghi vào manifest.

CPG binary chỉ tái sử dụng khi source hash khớp; xuất whole GraphML/AST/CFG/PDG vào cache mới. CPG thiếu mới chạy Joern parse. Cache CodeT5 positional và GraphVerb đã slice không được cắm vào baseline. Lần đầu có thể tốn thời gian cho embedding và graph của các demo; lần sau dùng cache có fingerprint.

In [3]:
if RUN_PREPARE:
    prepared = pipeline.prepare()
    print("Prepared:", prepared["status"])
else:
    print("Audit only. Set RUN_PREPARE=True to prepare all stages.")


Joern features: 12039 unique code records
Joern features: 25/12039
Joern features: 50/12039
Joern features: 75/12039
Joern features: 100/12039
Joern features: 125/12039
Joern features: 150/12039
Joern features: 175/12039
Joern features: 200/12039
Joern features: 225/12039
Joern features: 250/12039
Joern features: 275/12039
Joern features: 300/12039
Joern features: 325/12039
Joern features: 350/12039
Joern features: 375/12039
Joern features: 400/12039
Joern features: 425/12039
Joern features: 450/12039
Joern features: 475/12039
Joern features: 500/12039
Joern features: 525/12039
Joern features: 550/12039
Joern features: 575/12039
Joern features: 600/12039
Joern features: 625/12039
Joern features: 650/12039
Joern features: 675/12039
Joern features: 700/12039
Joern features: 725/12039
Joern features: 750/12039
Joern features: 775/12039
Joern features: 800/12039
Joern features: 825/12039
Joern features: 850/12039
Joern features: 875/12039
Joern features: 900/12039
Joern features: 925/12039

## 3. Freeze cấu hình và prompt

Sau khi xem prompt/manifest trên validation, chốt cấu hình. Nếu chuyển sang test, chạy audit và prepare trên toàn bộ test rồi freeze artefact đó. Freeze kiểm tra drift ở inference; nó không biến test đã dùng trước đây thành test mới.

In [4]:
if FREEZE:
    frozen = pipeline.freeze()
    print("Frozen:", pipeline.output / "freeze.json")


Frozen: /mnt/c/MY-FILES/LAB-THAY-THO/Graduation-Project/Ablation_V2/Reproduced-Code-Based/devign/test/freeze.json


## 4. Inference và evaluation chung

Chạy sau prepare + freeze. Thiết lập `VERTEX_PROJECT` và `VERTEX_LOCATION` trong môi trường trước khi mở notebook. `INFERENCE_WORKERS=10` giới hạn số request Vertex đồng thời; mỗi worker tái sử dụng client riêng, một luồng ghi checkpoint khi từng request hoàn thành. Model, temperature, seed, max tokens và retry lấy trực tiếp từ contract. Checkpoint chỉ tái sử dụng nếu prompt và cấu hình inference trùng. File kết quả cuối giữ thứ tự target; label mục tiêu chỉ dùng tại evaluator.

In [5]:
if RUN_INFERENCE:
    metrics = pipeline.infer()
    print(json.dumps(metrics, ensure_ascii=False, indent=2))
else:
    print("No API requests. RUN_INFERENCE=False.")


inference: 0/2732 cached; 10 Vertex workers
inference: 25/2732
inference: 50/2732
inference: 75/2732
inference: 100/2732
inference: 125/2732
inference: 150/2732
inference: 175/2732
inference: 200/2732
inference: 225/2732
inference: 250/2732
inference: 275/2732
inference: 300/2732
inference: 325/2732
inference: 350/2732
inference: 375/2732
inference: 400/2732
inference: 425/2732
inference: 450/2732
inference: 475/2732
inference: 500/2732
inference: 525/2732
inference: 550/2732
inference: 575/2732
inference: 600/2732
inference: 625/2732
inference: 650/2732
inference: 675/2732
inference: 700/2732
inference: 725/2732
inference: 750/2732
inference: 775/2732
inference: 800/2732
inference: 825/2732
inference: 850/2732
inference: 875/2732
inference: 900/2732
inference: 925/2732
inference: 950/2732
inference: 975/2732
inference: 1000/2732
inference: 1025/2732
inference: 1050/2732
inference: 1075/2732
inference: 1100/2732
inference: 1125/2732
inference: 1150/2732
inference: 1175/2732
inference: 

## Cách diễn giải

So sánh Full với bản tái lập này cho tác động tổng hợp của phương pháp cải tiến trong điều kiện contract. Leave-one-out hiện tại trả lời đóng góp từng factor trong Full. Hai phép so sánh có mục đích khác nhau. Tắt mọi No-C hiện có không tự khôi phục GRACE: graph gốc phải còn, retrieval gốc phải được thay lại, và metric contract vẫn giữ.